# Career Knowledge Assistant — Cohere RAG

Cloud-based version of this RAG pipeline: **Cohere** for embeddings + chat
(instead of local Ollama/MiniLM), **FAISS + BM25** hybrid search, built to
mirror `core/rag.py` exactly so this notebook and the deployable Streamlit
app (`app.py`) never drift out of sync.

**Prerequisite:** copy `.env.example` to `.env` in the project root and set
a real `COHERE_API_KEY` (free trial key at
[dashboard.cohere.com/api-keys](https://dashboard.cohere.com/api-keys))
before running the cells below. The key is read server-side only — see
`core/rag.py:resolve_secret()` — and is never printed by this notebook.

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolve the project root whether this notebook is run from the repo root
# or elsewhere, so paths and the `core` package import work on any machine.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir() and (PROJECT_ROOT.parent / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")  # no-op if .env doesn't exist yet

# Reuse the exact same engine app.py uses -- nothing in this notebook
# reimplements ingestion/retrieval/generation, it only calls core.rag.
from core.rag import (
    extract_text_from_pdfs, clean_document_text, build_chunks,
    build_index, retrieve, generate_answer, get_cohere_client,
    EMBED_MODEL, CHAT_MODEL,
)

DATA_DIR = PROJECT_ROOT / "data"

# Raises a clear RuntimeError (and never prints the key itself) if
# COHERE_API_KEY isn't set -- see core/rag.py:resolve_secret().
cohere_client = get_cohere_client()
print(f"✓ Cohere client ready. Embed model: {EMBED_MODEL} | Chat model: {CHAT_MODEL}")

records = extract_text_from_pdfs(DATA_DIR)
print(f"\nExtracted {len(records)} raw page(s) from {DATA_DIR}")

✓ Cohere client ready. Embed model: embed-english-v3.0 | Chat model: command-r-08-2024

Extracted 19 raw page(s) from C:\Users\Ahmed\Desktop\NRAG\data


In [3]:
records[0]

PageRecord(file_name='01_Resume_Writing_Best_Practices.pdf', page_number=1, text='Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actually changed with AI screening\n\x7f\nSemantic matching, not just keyword matching. Modern AI screening understands th

In [4]:
# clean_document_text lives in core/rag.py (shared with app.py) -- this
# cell just demonstrates what it does to one page's raw extracted text.
sample_raw = records[0].text
sample_cleaned = clean_document_text(sample_raw)

print("--- before cleaning (first 400 chars) ---")
print(sample_raw[:400])
print("\n--- after cleaning (first 400 chars) ---")
print(sample_cleaned[:400])

--- before cleaning (first 400 chars) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.
1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring 

--- after cleaning (first 400 chars) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.

1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring


In [5]:
for r in records[:3]:
    print(f"--- {r.file_name} (page {r.page_number}) ---")
    print(clean_document_text(r.text)[:400])
    print()

--- 01_Resume_Writing_Best_Practices.pdf (page 1) ---
Resume Writing Best Practices
 CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition
Target reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening
and a human recruiter's 7-second skim.

1. The Two Readers Every Resume Has
Every resume submitted online today is read twice: once by software (an Applicant Tracking System, often
layered with an AI scoring

--- 01_Resume_Writing_Best_Practices.pdf (page 2) ---
Core Competencies / Skills -- a compact grid of 6-10 keywords pulled directly from the kind of job
descriptions you're targeting (see the companion guide on analyzing job descriptions).

Professional Experience -- reverse-chronological, with impact-driven bullets (see Section 3).

Education & Certifications -- degree, institution, graduation year (omit GPA unless above 3.5 and
you're early-career)

--- 01_Resume_Writing_Best_Practices.pdf (page 3) ---
Mirror the job description's langu

In [6]:
# build_chunks() re-runs extraction + cleaning internally and splits into
# overlapping, content-hash-identified chunks -- same function app.py uses.
chunks = build_chunks(DATA_DIR)
print(f"✓ Generated {len(chunks)} cohesive chunks with context overlap.")

✓ Generated 73 cohesive chunks with context overlap.


In [7]:
chunks[:3]

[Chunk(chunk_id='01_Resume_Writing_Best_Practices.pdf_p1_77b0c04da0e7f9e6', file_name='01_Resume_Writing_Best_Practices.pdf', page_number=1, text="Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim."),
 Chunk(chunk_id='01_Resume_Writing_Best_Practices.pdf_p1_8eaf2df07ded7dce', file_name='01_Resume_Writing_Best_Practices.pdf', page_number=1, text='1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a 

In [8]:
# This calls the Cohere Embed API (batched, <=96 texts/call) to embed every
# chunk, then builds an in-memory FAISS (dense) + BM25 (sparse) index --
# the exact RagIndex class app.py caches at startup. No vector DB file is
# written to disk; re-running this cell just rebuilds the same index.
index = build_index(DATA_DIR, cohere_client)
print(f"✓ Indexed {len(index)} chunks via Cohere ({EMBED_MODEL}) + FAISS + BM25.")

✓ Indexed 73 chunks via Cohere (embed-english-v3.0) + FAISS + BM25.


In [9]:
def search_and_display(query: str, top_k: int = 4):
    """Pretty-print hybrid search results for quick manual inspection.

    retrieve() (from core.rag) fuses FAISS vector search with BM25 keyword
    search via Reciprocal Rank Fusion and drops anything neither method is
    confident about -- see core/rag.py for the full explanation.
    """
    print(f"\nSearching for: '{query}'")
    print("=" * 70)

    hits = retrieve(index, query, top_k=top_k)
    if not hits:
        print("No sufficiently relevant chunks found (vector + BM25 both below threshold).")
        return

    for i, hit in enumerate(hits, start=1):
        print(
            f"Result #{i} | vector={hit['vector_sim']}% bm25={hit['bm25_score']} "
            f"| Source: {hit['file_name']} (Page {hit['page_number']})"
        )
        print("-" * 70)
        print(hit["text"])
        print("=" * 70)

test_query = "How should I structure my resume bullet points to show measurable impact?"
search_and_display(test_query, top_k=4)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | vector=48.77% bm25=7.9 | Source: 02_How_to_Analyze_a_Job_Description.pdf (Page 2)
----------------------------------------------------------------------
5. Turning the Analysis Into Action

Rewrite your resume's Core Competencies section using the JD's own terminology for skills you
genuinely have.

Reorder your bullet points within each role so the most JD-relevant achievements appear first.

Draft 2-3 STAR stories (see the Behavioral Interview guide) that map directly to the top 3 requirements
in the posting.

If applying via a portal, paste your tailored resume and the JD into a keyword-comparison tool if one is
available to catch obvious gaps before submitting.
Result #2 | vector=55.9% bm25=4.92 | Source: 01_Resume_Writing_Best_Practices.pdf (Page 2)
----------------------------------------------------------------------
Projects / Portfolio (optional but increasingly expected in 

In [10]:
# generate_answer() (core.rag) is stateless -- it takes conversation
# history as a plain argument rather than owning global state, the same
# contract app.py uses with st.session_state.messages. This notebook just
# keeps a simple list and a couple of convenience wrappers around it.
conversation_history: list[dict] = []

def reset_conversation():
    conversation_history.clear()

def ask(question: str, top_k: int = 4) -> dict:
    result = generate_answer(index, question, conversation_history, top_k=top_k)
    conversation_history.append({"role": "user", "content": question})
    conversation_history.append({"role": "assistant", "content": result["answer"]})
    return result

In [11]:
# ====================================================
# End-to-end test
# ====================================================
reset_conversation()
test_query = "How should I structure my resume bullet points to show measurable impact?"

result = ask(test_query)

print("\n" + "=" * 70)
print("🤖 Final RAG Response:")
print("=" * 70)
print(result["answer"])

if result["cited_sources"]:
    print("\n📎 Cited sources (from Cohere's native citations, not regex-parsed):")
    for s in result["cited_sources"]:
        print(f"   - {s['file_name']} (Page {s['page_number']})")
elif result["sources"]:
    print("\n⚠️  The model answered without citing a specific retrieved source.")


🤖 Final RAG Response:
To structure your resume bullet points to show measurable impact, you should:

- Use the X-Y-Z formula: Accomplished [X], measured by [Y], by doing [Z]. For example: "Reduced customer churn by 18% (Y) by redesigning the onboarding email sequence (Z), resulting in $240K in retained annual revenue (X)."
- Lead every bullet with a strong action verb (e.g. led, built, reduced, automated, negotiated) rather than "Responsible for".
- Ensure every bullet has a measurable outcome or concrete scope.
- Include tools you used, the scale of the work, and measurable outcomes.
- Avoid generic skill lists and walls of buzzwords.

📎 Cited sources (from Cohere's native citations, not regex-parsed):
   - 01_Resume_Writing_Best_Practices.pdf (Page 1)
   - 01_Resume_Writing_Best_Practices.pdf (Page 2)
   - 01_Resume_Writing_Best_Practices.pdf (Page 3)


In [12]:
# ====================================================
# Demo: conversation memory + the "I don't know" relevance gate
# ====================================================
reset_conversation()

for question in [
    "What's a good salary negotiation tactic?",           # turn 1
    "Can you give me one more tip like that?",             # turn 2 -- needs memory of turn 1
    "What is the boiling point of water on Mars?",         # turn 3 -- off-topic, should refuse
]:
    result = ask(question)
    print("\n" + "=" * 70)
    print(f"🤖 Q: {question}")
    print("=" * 70)
    print(result["answer"])
    if result["cited_sources"]:
        print("\n📎 Cited:", ", ".join(f"{s['file_name']} p{s['page_number']}" for s in result["cited_sources"]))


🤖 Q: What's a good salary negotiation tactic?
Here are some tips for negotiating salary:

- **Preparation:** Research the market rate from multiple sources and factor in your specific variables, such as location, company size/stage, years of experience, and specialized/high-demand skills.
- **Know your number:** Decide your target, your walk-away minimum, and an ideal number above target before any conversation starts.
- **Understand the full compensation package:** Consider not just base salary, but also bonus structure, equity/stock, retirement matching, health benefits, remote/flexibility policy, learning budget, and PTO.
- **Let them anchor first:** Whoever states a number first gives away information, so redirect politely or give a well-researched range rather than a single figure.
- **Negotiate the whole package:** Don't just focus on base salary; other elements like sign-on bonus, equity refresh, start date, or title can often still be negotiated.
- **Always counter:** A first 